In [ ]:
# Instalando depndencias
!pip install -q transformers datasets evaluate accelerate scikit-learn matplotlib

In [ ]:
# Imports
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    ConfusionMatrixDisplay
)

In [ ]:
# Verificar uso de GPU
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Dispositivo:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Carregar modelo e tokenizer
model_name = "nlptown/bert-base-multilingual-uncased-sentiment"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device)

In [ ]:
# Carregar dataset
dataset = load_dataset("yelp_review_full")

dataset

In [ ]:
# Explorar dataset
print(dataset["train"][0])

df = pd.DataFrame(dataset["train"].select(range(10)))
df

In [ ]:
# Mapear labels para estrelas
label_map = {
    0: "1 star",
    1: "2 stars",
    2: "3 stars",
    3: "4 stars",
    4: "5 stars"
}

def label_to_stars(label):
    return label_map[label]

for i in range(5):
    print("Texto:", dataset["train"][i]["text"][:300])
    print("Label:", dataset["train"][i]["label"])
    print("Estrelas:", label_to_stars(dataset["train"][i]["label"]))
    print("-" * 80)

In [ ]:
# Distribuição das classes
sample_df = pd.DataFrame(dataset["train"].select(range(20000)))
sample_df["stars"] = sample_df["label"].map(label_map)

sample_df["stars"].value_counts().sort_index().plot(kind="bar")

plt.title("Distribuição das Classes")
plt.xlabel("Classe")
plt.ylabel("Quantidade")
plt.show()

In [ ]:
# Criar amostras pequenas para Colab
train_sample = dataset["train"].shuffle(seed=42).select(range(500))
validation_sample = dataset["test"].shuffle(seed=42).select(range(200))
test_sample = dataset["test"].shuffle(seed=123).select(range(200))

In [ ]:
# Tokenização
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=128
    )

tokenized_train = train_sample.map(tokenize_function, batched=True)
tokenized_validation = validation_sample.map(tokenize_function, batched=True)
tokenized_test = test_sample.map(tokenize_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
# Métricas
def compute_metrics(eval_pred):
    # logits = recebe as saídas do modelo.
    # labels = recebe os valores corretos do dataset.
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted"
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [ ]:
# Avaliar modelo antes do fine-tuning
initial_args = TrainingArguments(
    # Define a pasta onde resultados temporários podem ser salvos.
    output_dir="./initial_results",
    # Define que o modelo vai avaliar 16 textos por vez.
    per_device_eval_batch_size=16,
    # Desativa integração com ferramentas externas de log, como WandB.
    report_to="none"
)

# Apesar do nome ser Trainer, ele também serve para avaliar modelos.
initial_trainer = Trainer(
    # Informa qual modelo será avaliado.
    model=model,
    # Passa as configurações criadas antes.
    args=initial_args,
    # Informa qual dataset será usado para avaliação. (Ja tokenizado)
    eval_dataset=tokenized_test,
    # Cuida de organizar os dados em lotes, preenchendo textos menores com padding quando necessário.
    data_collator=data_collator,
    # Informa qual função será usada para calcular métricas como:
    compute_metrics=compute_metrics
)

# # Aqui sim a avaliação é executada.
# initial_results = initial_trainer.evaluate()

# initial_results

In [ ]:
# Matriz de confusão antes do fine-tuning
initial_predictions = initial_trainer.predict(tokenized_test)

y_true = initial_predictions.label_ids
y_pred = np.argmax(initial_predictions.predictions, axis=-1)

cm = confusion_matrix(y_true, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["1 star", "2 stars", "3 stars", "4 stars", "5 stars"]
)

disp.plot(cmap=None)
plt.title("Matriz de Confusão - Antes do Fine-tuning")
plt.show()

In [ ]:
# Gráfico de métricas antes do fine-tuning

accuracy = accuracy_score(y_true, y_pred)

precision, recall, f1, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    average="weighted"
)

metrics_before = {
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "f1": f1
}

plt.bar(metrics_before.keys(), metrics_before.values())
plt.title("Métricas Antes do Fine-tuning")
plt.ylim(0, 1)
plt.show()

metrics_before

In [ ]:
# Configurar fine-tuning
training_args = TrainingArguments(
    output_dir="./sentiment_model_finetuned",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none"
)

In [ ]:
# Treinar modelo
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
# Matriz de confusão depois do fine-tuning
final_predictions = trainer.predict(tokenized_test)

y_true_final = final_predictions.label_ids
y_pred_final = np.argmax(final_predictions.predictions, axis=-1)

cm_final = confusion_matrix(y_true_final, y_pred_final)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_final,
    display_labels=["1 star", "2 stars", "3 stars", "4 stars", "5 stars"]
)

disp.plot(cmap=None)
plt.title("Matriz de Confusão - Depois do Fine-tuning")
plt.show()

In [ ]:
# Métricas depois do fine-tuning

accuracy_after = accuracy_score(y_true_final, y_pred_final)

precision_after, recall_after, f1_after, _ = precision_recall_fscore_support(
    y_true_final,
    y_pred_final,
    average="weighted"
)

metrics_after = {
    "accuracy": accuracy_after,
    "precision": precision_after,
    "recall": recall_after,
    "f1": f1_after
}

metrics_after

In [ ]:
comparison_df = pd.DataFrame({
    "Before Fine-tuning": metrics_before,
    "After Fine-tuning": metrics_after
})

comparison_df

In [ ]:
comparison_df.plot(kind="bar")

plt.title("Comparação Antes vs Depois do Fine-tuning")
plt.ylabel("Valor")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Função para consultar frases
def predict_sentiment(text):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)
    predicted_class = torch.argmax(probabilities, dim=-1).item()
    confidence = probabilities[0][predicted_class].item()

    return {
        "text": text,
        "predicted_label": predicted_class,
        "sentiment": label_to_stars(predicted_class),
        "confidence": round(confidence, 4)
    }

In [ ]:
# Testar frases novas
examples = [
    "The food was amazing and the service was perfect!",
    "The restaurant was terrible. I will never come back.",
    "It was okay, nothing special.",
    "The coffee was good, but the service was slow.",
    "I loved this place. Everything was excellent!"
]

for example in examples:
    result = predict_sentiment(example)
    print(result)

In [ ]:
# Salvar modelo treinado
model.save_pretrained("./modelo_yelp_finetuned")
tokenizer.save_pretrained("./modelo_yelp_finetuned")